In [ ]:
cell_str=r'''
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

// Necessary libraries
#include <cuda_runtime.h>
#include <cublas_v2.h>

int main(int argc, char **argv) {

    if (argc < 2) {
        fprintf(stderr, "Error: missing argument in %s \n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);

    // Host allocation
    size_t bytes = n * n * sizeof(float);
    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    // Initialization
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    // Device allocation: cudaMalloc
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Data transferring from Host to Device
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Context creation for the cuBLAS computation
    cublasHandle_t handle;
    cublasCreate(&handle);

    // Parameter setting, as usual
    float alpha = 1.0f;
    float beta = 0.0f;

    // CUDA Timing events
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Start Timing
    cudaEventRecord(start);

    // cuBLAS computation: launch of the function "Sgemm" (Single precision GEneral Matrix Multiply)
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N,
                n, n, n,
                &alpha,
                d_b, n,
                d_a, n,
                &beta,
                d_c, n);

    // Stop Timing
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("cuBLAS Time for n=%d: %.4f seconds\n", n, milliseconds / 1000.0f);

    cublasDestroy(handle);

    // Data transferring (results) from Device to Host
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);


    // Check
    FILE *f = fopen("mat-res.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int sample = (n < 100) ? n : 10;
        for (int i = 0; i < sample; i++) {
            for (int j = 0; j < sample; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    // Memory deallocation: both Host and Device
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_matrixmult_cuBLAS.cu', 'w') as f:
    f.write(cell_str)

In [ ]:
!nvcc -arch=sm_75 -O3 -lcublas cuda_matrixmult_cuBLAS.cu -o cuda_matrixmult_cuBLAS

In [ ]:
!nvprof ./cuda_matrixmult_cuBLAS 20000

==5687== NVPROF is profiling process 5687, command: ./cuda_matrixmult_cuBLAS 20000
cuBLAS Time for n=20000: 3.4423 seconds
==5687== Profiling application: ./cuda_matrixmult_cuBLAS 20000
==5687== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   75.05%  3.38335s         1  3.38335s  3.38335s  3.38335s  volta_sgemm_128x128_nn
                   16.04%  723.34ms         2  361.67ms  346.17ms  377.17ms  [CUDA memcpy HtoD]
                    8.91%  401.51ms         1  401.51ms  401.51ms  401.51ms  [CUDA memcpy DtoH]
      API calls:   70.94%  3.38336s         1  3.38336s  3.38336s  3.38336s  cudaEventSynchronize
                   23.62%  1.12657s         3  375.52ms  347.21ms  401.94ms  cudaMemcpy
                    3.97%  189.42ms         6  31.569ms  21.625us  188.73ms  cudaMalloc
                    0.68%  32.505ms         1  32.505ms  32.505ms  32.505ms  cudaGetSymbolAddress
                    0.27%  12.934ms      